In [ ]:
!pip install transformers datasets tensorflow scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import BertTokenizer

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/chatbot/medical_nonmedical_queries_huge.csv")

# Convert labels (Medical = 1, Non-Medical = 0)
df['Category'] = df['Category'].map({'Medical': 1, 'Non-Medical': 0})

# Train-test split (80-20)
from sklearn.model_selection import train_test_split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["Query"].tolist(), df["Category"].tolist(), test_size=0.2, random_state=42
)

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize queries
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
# Convert to TensorFlow Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_labels
)).batch(16)  # Batch size 16

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    test_labels
)).batch(16)  # Batch size 16


In [ ]:
from transformers import TFBertForSequenceClassification

# Load BERT with 2 output classes
model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Compile model with optimizer, loss, and metrics
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


In [ ]:
# Train the model
model.fit(train_dataset, validation_data=test_dataset, epochs=10)


Epoch 1/10
450/450 [==============================] - 98s 112ms/step - loss: 0.0347 - accuracy: 0.9919 - val_loss: 2.7680e-04 - val_accuracy: 1.0000
Epoch 2/10
450/450 [==============================] - 45s 100ms/step - loss: 2.2726e-04 - accuracy: 1.0000 - val_loss: 8.3366e-05 - val_accuracy: 1.0000
Epoch 3/10
450/450 [==============================] - 46s 103ms/step - loss: 9.8195e-05 - accuracy: 1.0000 - val_loss: 4.5364e-05 - val_accuracy: 1.0000
Epoch 4/10
450/450 [==============================] - 47s 105ms/step - loss: 5.7809e-05 - accuracy: 1.0000 - val_loss: 2.9004e-05 - val_accuracy: 1.0000
Epoch 5/10
450/450 [==============================] - 46s 102ms/step - loss: 3.8218e-05 - accuracy: 1.0000 - val_loss: 2.0220e-05 - val_accuracy: 1.0000
Epoch 6/10
450/450 [==============================] - 45s 101ms/step - loss: 2.7304e-05 - accuracy: 1.0000 - val_loss: 1.4766e-05 - val_accuracy: 1.0000
Epoch 7/10
450/450 [==============================] - 48s 106ms/step - loss: 2.0158e-0

In [ ]:
# Evaluate model accuracy on test data
loss, accuracy = model.evaluate(test_dataset)
print(f"Test Accuracy: {accuracy:.4f}")


113/113 [==============================] - 4s 34ms/step - loss: 5.4516e-06 - accuracy: 1.0000
Test Accuracy: 1.0000


In [ ]:
# Save trained model
model.save_pretrained("bert_medical_classifier_tf")
tokenizer.save_pretrained("bert_medical_classifier_tf")

# Load model later
from transformers import TFBertForSequenceClassification, BertTokenizer
model = TFBertForSequenceClassification.from_pretrained("bert_medical_classifier_tf")
tokenizer = BertTokenizer.from_pretrained("bert_medical_classifier_tf")


Some layers from the model checkpoint at bert_medical_classifier_tf were not used when initializing TFBertForSequenceClassification: ['dropout_113']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at bert_medical_classifier_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [ ]:
from transformers import TFBertForSequenceClassification, BertTokenizer
import tensorflow as tf

# Load trained model
model = TFBertForSequenceClassification.from_pretrained("bert_medical_classifier_tf")
tokenizer = BertTokenizer.from_pretrained("bert_medical_classifier_tf")


Some layers from the model checkpoint at bert_medical_classifier_tf were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at bert_medical_classifier_tf.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [ ]:
def classify_query(query):
    # Tokenize input text
    inputs = tokenizer(query, return_tensors="tf", truncation=True, padding=True, max_length=128)

    # Get model predictions
    outputs = model(**inputs)

    # Convert logits to class prediction
    predicted_class = tf.argmax(outputs.logits, axis=1).numpy()[0]

    # Return classification result
    return "Medical" if predicted_class == 1 else "Non-Medical"


In [ ]:
while True:
    user_query = input("Enter your query (or type 'exit' to stop): ")

    if user_query.lower() == "exit":
        print("Exiting the program. Goodbye!")
        break  # Stop the loop

    prediction = classify_query(user_query)  # Call the classification function
    print(f"Prediction: {prediction}\n")


Enter your query (or type 'exit' to stop): how to cure headache
Prediction: Medical

Enter your query (or type 'exit' to stop): how to cook biryani
Prediction: Non-Medical

Enter your query (or type 'exit' to stop): how to start a bike 
Prediction: Non-Medical

Enter your query (or type 'exit' to stop): what is bike 
Prediction: Non-Medical

Enter your query (or type 'exit' to stop): how to swim 
Prediction: Non-Medical

Enter your query (or type 'exit' to stop): how to kill bacteria
Prediction: Medical

Enter your query (or type 'exit' to stop): how to run
Prediction: Non-Medical

Enter your query (or type 'exit' to stop): how to run after injury 
Prediction: Medical

Enter your query (or type 'exit' to stop): who is doctor
Prediction: Non-Medical

Enter your query (or type 'exit' to stop): acne
Prediction: Medical

Enter your query (or type 'exit' to stop): itchy
Prediction: Non-Medical

Enter your query (or type 'exit' to stop): how to cure itchy scalp
Prediction: Medical

Enter you

In [ ]:
!pip install transformers datasets torch scikit-learn pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import torch

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/chatbot/medical_nonmedical_queries_huge.csv")  # Use your dataset

# Convert labels (Medical = 1, Non-Medical = 0)
df['Category'] = df['Category'].map({'Medical': 1, 'Non-Medical': 0})

# Train-test split (80-20)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["Query"].tolist(), df["Category"].tolist(), test_size=0.2, random_state=42
)

# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize queries
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

# Convert labels to tensors
train_labels = torch.tensor(train_labels)
test_labels = torch.tensor(test_labels)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
from torch.utils.data import Dataset

class QueryDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

# Create dataset objects
train_dataset = QueryDataset(train_encodings, train_labels)
test_dataset = QueryDataset(test_encodings, test_labels)


In [ ]:
from transformers import BertForSequenceClassification

# Load BERT with 2 output classes (Medical & Non-Medical)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.000200,0.000123
2,0.000000,0.000030
3,0.000000,0.000022


TrainOutput(global_step=1350, training_loss=0.03447556805917648, metrics={'train_runtime': 181.789, 'train_samples_per_second': 118.819, 'train_steps_per_second': 7.426, 'total_flos': 188699960016000.0, 'train_loss': 0.03447556805917648, 'epoch': 3.0})

In [ ]:
# Save trained model
model.save_pretrained("bert_medical_classifier_pytorch")
tokenizer.save_pretrained("bert_medical_classifier_pytorch")

# Load model later
from transformers import BertForSequenceClassification, BertTokenizer

model = BertForSequenceClassification.from_pretrained("bert_medical_classifier_pytorch")
tokenizer = BertTokenizer.from_pretrained("bert_medical_classifier_pytorch")
